# Notebook-first application walkthrough

**Problem / objective:** Train a real neural network on bank-marketing data and compare it against a simpler baseline instead of assuming deep learning wins.

**Decision / solution:** Choose the model and threshold from measured validation performance, calibration and campaign economics, not architecture complexity.

This front section is intentionally analysis-first. It uses direct notebook code for inspection, EDA, visualisation and evidence review. The original notebook work is preserved below, followed by modular production code where that adds engineering evidence.


In [ ]:
from pathlib import Path
import json
import os
import subprocess
import sys
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings('ignore')
PROJECT_SLUG = 'deep_learning_marketing_response'
ROOT = Path.cwd()
if not (ROOT / 'projects').exists():
    candidate = ROOT.parent.parent if ROOT.name == PROJECT_SLUG else ROOT
    if (candidate / 'projects').exists():
        ROOT = candidate
PROJECT = ROOT / 'projects' / PROJECT_SLUG
if not PROJECT.exists() and Path.cwd().name == PROJECT_SLUG:
    PROJECT = Path.cwd()
    ROOT = PROJECT.parent.parent
assert PROJECT.exists(), f'Project directory not found: {PROJECT}'
print('Repository root:', ROOT.resolve())
print('Project:', PROJECT.resolve())


## 1. Find the real data and retained evidence

Instead of hiding the dataset behind a helper function, start by seeing what the project actually ships: raw/small data, fixtures, outputs, results and verified evidence. External large datasets remain reproducibly downloadable from the documented source.


In [ ]:
candidate_files = []
for pattern in ('*.csv', '*.parquet', '*.json', '*.tsv', '*.txt'):
    candidate_files.extend(PROJECT.rglob(pattern))
verified_dir = ROOT / 'verified' / PROJECT_SLUG
if verified_dir.exists():
    for pattern in ('*.csv', '*.parquet', '*.json', '*.tsv', '*.txt'):
        candidate_files.extend(verified_dir.rglob(pattern))
candidate_files = sorted({p.resolve() for p in candidate_files if p.is_file()})
file_inventory = pd.DataFrame({
    'file': [str(p.relative_to(ROOT)) if ROOT in p.parents else str(p) for p in candidate_files],
    'suffix': [p.suffix.lower() for p in candidate_files],
    'size_kb': [round(p.stat().st_size / 1024, 1) for p in candidate_files],
})
display(file_inventory.head(40))
print(f'Inspectable local data/evidence files: {len(file_inventory):,}')


## 2. Direct tabular data audit

The code below deliberately avoids a project-specific wrapper. It opens the first sensible local tabular asset, shows its schema and quality profile, and makes the data issues visible before modelling. If the full raw dataset is external, run the project's documented download cell/entry point first and rerun this section.


In [ ]:
tabular_candidates = [p for p in candidate_files if p.suffix.lower() in {'.csv', '.tsv', '.parquet'}]
preferred = [p for p in tabular_candidates if not any(token in p.name.lower() for token in ('metric', 'summary', 'verification'))]
tabular_path = (preferred or tabular_candidates or [None])[0]
df = None
if tabular_path is not None:
    if tabular_path.suffix.lower() == '.parquet':
        df = pd.read_parquet(tabular_path)
    else:
        sep = '\t' if tabular_path.suffix.lower() == '.tsv' else ','
        df = pd.read_csv(tabular_path, sep=sep, nrows=200_000)
    print('Loaded:', tabular_path)
    print('Shape:', df.shape)
    display(df.head())
    audit = pd.DataFrame({
        'dtype': df.dtypes.astype(str),
        'missing': df.isna().sum(),
        'missing_pct': (100 * df.isna().mean()).round(2),
        'unique': df.nunique(dropna=False),
    }).sort_values(['missing_pct', 'unique'], ascending=[False, False])
    display(audit.head(30))
    print('Duplicate rows:', int(df.duplicated().sum()))
else:
    print('No local CSV/TSV/Parquet found yet. Use the project README/run path to download or build the documented dataset, then rerun this audit.')


## 3. Exploratory data analysis and visualisation

These plots are intentionally created in the notebook rather than described in prose. They expose distribution, missingness, scale, category balance and numeric relationships before any final model decision.


In [ ]:
if df is not None and len(df):
    missing_pct = (100 * df.isna().mean()).sort_values(ascending=False).head(20)
    missing_pct = missing_pct[missing_pct > 0]
    if len(missing_pct):
        plt.figure(figsize=(10, 4))
        missing_pct.plot(kind='bar')
        plt.title('Missing values by feature (%)')
        plt.ylabel('Missing %')
        plt.xticks(rotation=60, ha='right')
        plt.tight_layout()
        plt.show()

    numeric_cols = df.select_dtypes(include=np.number).columns.tolist()[:8]
    for col in numeric_cols:
        series = pd.to_numeric(df[col], errors='coerce').dropna()
        if len(series):
            plt.figure(figsize=(8, 4))
            plt.hist(series, bins=30, alpha=0.8)
            plt.axvline(series.median(), linestyle='--', label=f'median={series.median():.2f}')
            plt.title(f'Distribution: {col}')
            plt.xlabel(col)
            plt.ylabel('Count')
            plt.legend()
            plt.tight_layout()
            plt.show()

    categorical_cols = [c for c in df.columns if c not in numeric_cols and df[c].nunique(dropna=False) <= 30][:4]
    for col in categorical_cols:
        counts = df[col].fillna('<missing>').astype(str).value_counts().head(15)
        plt.figure(figsize=(9, 4))
        counts.sort_values().plot(kind='barh')
        plt.title(f'Top categories: {col}')
        plt.xlabel('Rows')
        plt.tight_layout()
        plt.show()

    if len(numeric_cols) >= 2:
        corr = df[numeric_cols].corr(numeric_only=True)
        plt.figure(figsize=(8, 6))
        image = plt.imshow(corr, vmin=-1, vmax=1, cmap='coolwarm')
        plt.colorbar(image, label='Correlation')
        plt.xticks(range(len(corr.columns)), corr.columns, rotation=60, ha='right')
        plt.yticks(range(len(corr.index)), corr.index)
        plt.title('Numeric correlation matrix')
        plt.tight_layout()
        plt.show()

    if len(numeric_cols) >= 2:
        x_col, y_col = numeric_cols[0], numeric_cols[-1]
        sample = df[[x_col, y_col]].dropna().sample(min(3000, len(df.dropna(subset=[x_col, y_col]))), random_state=42)
        if len(sample):
            plt.figure(figsize=(7, 5))
            plt.scatter(sample[x_col], sample[y_col], alpha=0.35, s=18)
            plt.xlabel(x_col)
            plt.ylabel(y_col)
            plt.title(f'{y_col} versus {x_col}')
            plt.tight_layout()
            plt.show()
else:
    print('Run the documented data-build/download path, then rerun this section to render raw-data EDA.')


## 4. Inspect the measured results, not just the code

A portfolio project is stronger when it retains evidence. This section reads machine-readable JSON/CSV outputs and turns scalar metrics into a quick visual comparison.


In [ ]:
json_files = [p for p in candidate_files if p.suffix.lower() == '.json']
metric_rows = []
for path in json_files[:30]:
    try:
        payload = json.loads(path.read_text(encoding='utf-8'))
    except Exception:
        continue
    stack = [('', payload)]
    while stack:
        prefix, value = stack.pop()
        if isinstance(value, dict):
            for key, child in value.items():
                stack.append((f'{prefix}.{key}' if prefix else str(key), child))
        elif isinstance(value, (int, float)) and not isinstance(value, bool) and np.isfinite(value):
            metric_rows.append({
                'file': str(path.relative_to(ROOT)) if ROOT in path.parents else str(path),
                'metric': prefix,
                'value': float(value),
            })
metrics_df = pd.DataFrame(metric_rows)
if len(metrics_df):
    display(metrics_df.head(40))
    plot_df = metrics_df[np.isfinite(metrics_df['value'])].copy()
    plot_df = plot_df[plot_df['value'].abs() < 1_000_000].head(20)
    if len(plot_df):
        labels = (plot_df['file'].str.split('/').str[-1] + ' :: ' + plot_df['metric']).tolist()
        plt.figure(figsize=(10, max(4, 0.35 * len(plot_df))))
        plt.barh(range(len(plot_df)), plot_df['value'])
        plt.yticks(range(len(plot_df)), labels)
        plt.title('Retained project metrics / evidence')
        plt.tight_layout()
        plt.show()
else:
    print('No scalar JSON evidence found. Run the project and retain metrics/results before treating it as complete.')


## 5. Reproduce the application

The notebook should be understandable without running anything, but a reviewer can reproduce the canonical application below. The switch is off by default so opening the notebook never triggers a long training job unexpectedly.


In [ ]:
RUN_PROJECT = False
entrypoint = PROJECT / 'run.py'
if RUN_PROJECT and entrypoint.exists():
    subprocess.run([sys.executable, str(entrypoint)], cwd=PROJECT, check=True)
elif entrypoint.exists():
    print(f'Reproduce with: cd {PROJECT} && {sys.executable} run.py')
else:
    print('This project uses a different documented entry point; see README.md in the project folder.')


## 6. Decision / solution

Choose the model and threshold from measured validation performance, calibration and campaign economics, not architecture complexity.

The final recommendation should be tied to the measured validation evidence and error analysis below. A model is not the solution by itself; the solution is the decision process built around it.


# Deep Learning Marketing Response — Neural Network Application

## Problem and objective
Train a real PyTorch neural network to estimate marketing-response probability, compare it with a classical baseline and translate predictions into a constrained outreach decision.


## Dataset and provenance
UCI Bank Marketing (`bank-additional-full.csv`) downloaded reproducibly from the UCI archive. The post-call `duration` field is removed because it would leak information unavailable at pre-contact decision time. This is an educational marketing benchmark, not a financial eligibility model.


In [ ]:
from run import load_data, audit_data, prepare_frame, temporal_like_split
raw = load_data()
audit_data(raw)


## Neural-network training and validation
The application builds numeric/categorical preprocessing, a logistic-regression baseline and a multi-layer PyTorch MLP with BatchNorm, GELU, dropout, class weighting, AdamW, learning-rate scheduling, gradient clipping and early stopping. Evaluation includes ROC-AUC, PR-AUC, F1, precision, recall, Brier score and log loss. The canonical full implementation is mirrored into this notebook by the portfolio workflow.


In [ ]:
data = prepare_frame(raw)
train, validation, test = temporal_like_split(data)
len(train), len(validation), len(test), data['y'].mean()


In [ ]:
from run import MarketingMLP
demo_network = MarketingMLP(input_dim=32)
demo_network


## Decision layer, results and limitations
The validation set selects a probability threshold subject to a maximum outreach rate. The test set is used once for final evidence, with calibration bins and subgroup error slices saved under `results/`. Production use requires consent/privacy controls, current data, cost-calibrated thresholds, drift monitoring and human review of campaign policy.

## Reproducibility
Run `python run.py`. The model checkpoint and fitted preprocessor are stored in `artifacts/`; tests are in `tests/test_neural_network.py`.


## Interview discussion
Be ready to explain the network architecture, BatchNorm/dropout, class weighting, AdamW, early stopping, why the logistic baseline matters, why the post-call duration field is leakage, how threshold selection controls campaign volume, and why better neural-network metrics do not automatically imply business value.


# Deeper exploratory analysis and retained evidence

These direct notebook cells extend the initial EDA with data-quality, scale, relationship, output and error diagnostics. They are intentionally visible here rather than hidden behind project helper functions.


In [ ]:
# Extended data-quality scorecard
if df is not None and len(df):
    quality_rows = []
    for col in df.columns:
        series = df[col]
        row = {
            'feature': col,
            'dtype': str(series.dtype),
            'rows': len(series),
            'missing': int(series.isna().sum()),
            'missing_pct': float(100 * series.isna().mean()),
            'unique': int(series.nunique(dropna=False)),
            'unique_pct': float(100 * series.nunique(dropna=False) / max(len(series), 1)),
        }
        if pd.api.types.is_numeric_dtype(series):
            values = pd.to_numeric(series, errors='coerce').dropna()
            if len(values):
                q1, q3 = values.quantile([0.25, 0.75])
                iqr = q3 - q1
                row.update({
                    'mean': float(values.mean()),
                    'median': float(values.median()),
                    'std': float(values.std()),
                    'p05': float(values.quantile(0.05)),
                    'p95': float(values.quantile(0.95)),
                    'skew': float(values.skew()),
                    'iqr_outliers': int(((values < q1 - 1.5*iqr) | (values > q3 + 1.5*iqr)).sum()),
                })
        quality_rows.append(row)
    deep_quality = pd.DataFrame(quality_rows)
    display(deep_quality.sort_values(['missing_pct','unique'], ascending=[False,False]).head(40))
    if 'iqr_outliers' in deep_quality:
        outlier_view = deep_quality.dropna(subset=['iqr_outliers']).sort_values('iqr_outliers', ascending=False).head(15)
        if len(outlier_view):
            plt.figure(figsize=(10,4))
            plt.bar(outlier_view['feature'], outlier_view['iqr_outliers'])
            plt.title('Potential IQR outliers by feature')
            plt.ylabel('Rows')
            plt.xticks(rotation=60, ha='right')
            plt.tight_layout()
            plt.show()
    card = deep_quality.sort_values('unique', ascending=False).head(20)
    plt.figure(figsize=(10,4))
    plt.bar(card['feature'], card['unique'])
    plt.title('Feature cardinality')
    plt.ylabel('Unique values')
    plt.xticks(rotation=60, ha='right')
    plt.tight_layout()
    plt.show()
    print('Constant columns:', deep_quality.loc[deep_quality['unique'] <= 1, 'feature'].tolist())
    print('High-missing columns:', deep_quality.loc[deep_quality['missing_pct'] >= 30, 'feature'].tolist())
    print('Possible identifier columns:', deep_quality.loc[deep_quality['unique_pct'] >= 95, 'feature'].tolist()[:20])
else:
    print('Materialise the documented dataset to run the extended data-quality scorecard.')


In [ ]:
# Numeric distributions, spread and strongest pairwise relationships
if df is not None and len(df):
    numeric_cols = df.select_dtypes(include=np.number).columns.tolist()[:12]
    for col in numeric_cols:
        values = pd.to_numeric(df[col], errors='coerce').dropna()
        if len(values) < 5:
            continue
        clipped = values.clip(values.quantile(0.01), values.quantile(0.99))
        plt.figure(figsize=(8,4))
        plt.hist(clipped, bins=35, alpha=0.82)
        plt.axvline(values.median(), linestyle='--', label=f'median={values.median():.3g}')
        plt.axvline(values.mean(), linestyle=':', label=f'mean={values.mean():.3g}')
        plt.title(f'Distribution: {col} (1st–99th percentile)')
        plt.xlabel(col)
        plt.ylabel('Rows')
        plt.legend()
        plt.tight_layout()
        plt.show()
        plt.figure(figsize=(8,3))
        plt.boxplot(values, vert=False, showfliers=True)
        plt.title(f'Spread / outliers: {col}')
        plt.xlabel(col)
        plt.tight_layout()
        plt.show()
    if len(numeric_cols) >= 2:
        corr = df[numeric_cols].corr(numeric_only=True)
        pairs = []
        for i, left in enumerate(corr.columns):
            for right in corr.columns[i+1:]:
                value = corr.loc[left, right]
                if pd.notna(value):
                    pairs.append({'feature_a': left, 'feature_b': right, 'correlation': float(value), 'abs_correlation': float(abs(value))})
        corr_pairs = pd.DataFrame(pairs).sort_values('abs_correlation', ascending=False) if pairs else pd.DataFrame()
        if len(corr_pairs):
            display(corr_pairs.head(20).round(4))
            for _, pair in corr_pairs.head(4).iterrows():
                sample = df[[pair['feature_a'], pair['feature_b']]].dropna()
                if len(sample) > 3000:
                    sample = sample.sample(3000, random_state=42)
                plt.figure(figsize=(7,5))
                plt.scatter(sample[pair['feature_a']], sample[pair['feature_b']], alpha=0.30, s=16)
                plt.xlabel(pair['feature_a'])
                plt.ylabel(pair['feature_b'])
                plt.title(f"{pair['feature_a']} vs {pair['feature_b']} (r={pair['correlation']:.2f})")
                plt.tight_layout()
                plt.show()
    categorical = [c for c in df.columns if 2 <= df[c].nunique(dropna=False) <= 20][:8]
    for col in categorical:
        counts = df[col].fillna('<missing>').astype(str).value_counts().head(20)
        shares = 100 * counts / counts.sum()
        display(pd.DataFrame({'rows': counts, 'share_pct': shares.round(2)}))
        plt.figure(figsize=(8,4))
        counts.sort_values().plot(kind='barh')
        plt.title(f'Category balance: {col}')
        plt.xlabel('Rows')
        plt.tight_layout()
        plt.show()
else:
    print('Materialise the documented dataset to run distribution diagnostics.')


In [ ]:
# Temporal coverage where date/time fields exist
if df is not None and len(df):
    time_cols = [c for c in df.columns if any(token in str(c).lower() for token in ('date','time','timestamp','datetime'))]
    print('Date/time candidates:', time_cols[:10])
    for col in time_cols[:4]:
        converted = pd.to_datetime(df[col], errors='coerce')
        valid = converted.dropna()
        if len(valid) >= max(10, int(0.25*len(df))):
            print(col, 'range:', valid.min(), '→', valid.max())
            monthly = valid.dt.to_period('M').value_counts().sort_index()
            if len(monthly) > 1:
                plt.figure(figsize=(10,4))
                plt.plot(monthly.index.astype(str), monthly.values, marker='o')
                plt.title(f'Rows over time: {col}')
                plt.ylabel('Rows')
                plt.xticks(rotation=70, ha='right')
                plt.tight_layout()
                plt.show()


## Retained outputs and error analysis

A strong portfolio keeps inspectable evidence. The cells below profile compact result tables and automatically detect prediction-like columns for residual or misclassification analysis.


In [ ]:
# Load compact result/evidence tables
result_tables = []
for base in [PROJECT/'results', PROJECT/'outputs', PROJECT/'artifacts', ROOT/'verified'/PROJECT_SLUG]:
    if not base.exists():
        continue
    for path in sorted(base.rglob('*')):
        if path.is_file() and path.suffix.lower() in {'.csv','.tsv','.parquet'} and path.stat().st_size < 25_000_000:
            try:
                if path.suffix.lower() == '.parquet':
                    table = pd.read_parquet(path)
                else:
                    table = pd.read_csv(path, sep='	' if path.suffix.lower() == '.tsv' else ',')
            except Exception as exc:
                print('Could not read', path.name, '-', exc)
                continue
            result_tables.append((path, table))
            print('
RESULT TABLE:', path.relative_to(ROOT) if ROOT in path.parents else path)
            print('shape=', table.shape)
            display(table.head(15))
            numeric = table.select_dtypes(include=np.number).columns.tolist()[:12]
            if numeric:
                display(table[numeric].describe().T.round(4))
print('Inspectable result tables:', len(result_tables))


In [ ]:
# Automatic regression/classification-style error diagnostics
actual_tokens = ('actual','target','truth','y_true','observed','label')
pred_tokens = ('prediction','predicted','forecast','y_pred')
confidence_tokens = ('confidence','probability','proba','risk','uncertainty')
for path, table in result_tables:
    actual_cols = [c for c in table.columns if any(token in str(c).lower() for token in actual_tokens)]
    pred_cols = [c for c in table.columns if any(token in str(c).lower() for token in pred_tokens)]
    conf_cols = [c for c in table.columns if any(token in str(c).lower() for token in confidence_tokens)]
    if actual_cols and pred_cols and len(table):
        actual_col = actual_cols[0]
        pred_col = next((c for c in pred_cols if c != actual_col), pred_cols[0])
        actual_num = pd.to_numeric(table[actual_col], errors='coerce')
        pred_num = pd.to_numeric(table[pred_col], errors='coerce')
        numeric_mask = actual_num.notna() & pred_num.notna()
        if numeric_mask.sum() >= 10:
            residual = actual_num[numeric_mask] - pred_num[numeric_mask]
            abs_error = residual.abs()
            print('
', path.name, '| MAE=', round(float(abs_error.mean()),5), '| RMSE=', round(float(np.sqrt(np.mean(residual**2))),5), '| bias=', round(float(residual.mean()),5))
            plt.figure(figsize=(7,5))
            plt.scatter(actual_num[numeric_mask], pred_num[numeric_mask], alpha=0.35, s=18)
            lo = min(actual_num[numeric_mask].min(), pred_num[numeric_mask].min())
            hi = max(actual_num[numeric_mask].max(), pred_num[numeric_mask].max())
            plt.plot([lo,hi],[lo,hi], linestyle='--')
            plt.xlabel(str(actual_col))
            plt.ylabel(str(pred_col))
            plt.title(f'Actual vs predicted — {path.name}')
            plt.tight_layout()
            plt.show()
            plt.figure(figsize=(7,4))
            plt.hist(residual, bins=30, alpha=0.82)
            plt.axvline(0, linestyle='--')
            plt.title(f'Residual distribution — {path.name}')
            plt.tight_layout()
            plt.show()
            worst_idx = abs_error.nlargest(min(15,len(abs_error))).index
            cols = list(dict.fromkeys([actual_col,pred_col]+conf_cols[:2]))
            worst = table.loc[worst_idx, cols].copy()
            worst['absolute_error'] = abs_error.loc[worst_idx].values
            display(worst.sort_values('absolute_error', ascending=False))
        else:
            agreement = table[actual_col].astype(str) == table[pred_col].astype(str)
            print('
', path.name, '| classification agreement=', round(float(agreement.mean()),4))
            if (~agreement).any():
                display(table.loc[~agreement, [actual_col,pred_col]+conf_cols[:2]].head(20))
    elif conf_cols:
        for col in conf_cols[:2]:
            values = pd.to_numeric(table[col], errors='coerce').dropna()
            if len(values) >= 10:
                plt.figure(figsize=(7,4))
                plt.hist(values, bins=30, alpha=0.82)
                plt.title(f'{col} distribution — {path.name}')
                plt.tight_layout()
                plt.show()


In [ ]:
# Display retained visual evidence from actual project runs
png_files = []
for base in [PROJECT/'results', PROJECT/'outputs', PROJECT/'artifacts', ROOT/'verified'/PROJECT_SLUG]:
    if base.exists():
        png_files.extend(sorted(base.rglob('*.png')))
print('Retained PNG figures:', len(png_files))
for path in png_files[:12]:
    try:
        image = plt.imread(path)
        plt.figure(figsize=(10,6))
        plt.imshow(image)
        plt.axis('off')
        plt.title(str(path.relative_to(ROOT)) if ROOT in path.parents else path.name)
        plt.tight_layout()
        plt.show()
    except Exception as exc:
        print('Could not display', path.name, '-', exc)


In [ ]:
# Reproducibility and evidence checklist
checks = [
    {'check':'README present', 'status':(PROJECT/'README.md').exists()},
    {'check':'Recruiter notebook present', 'status':(PROJECT/'project_notebook.ipynb').exists()},
    {'check':'Python implementation present', 'status':any(PROJECT.rglob('*.py'))},
    {'check':'Tests present', 'status':(PROJECT/'tests').exists() and any((PROJECT/'tests').rglob('test*.py'))},
    {'check':'Result/evidence files present', 'status':bool(candidate_files)},
    {'check':'Machine-readable JSON evidence', 'status':bool(json_files)},
    {'check':'Retained visual evidence', 'status':bool(png_files)},
]
checklist = pd.DataFrame(checks)
display(checklist)
print('Evidence checklist pass rate:', f"{100*checklist['status'].mean():.1f}%")
print('A failed item is a prompt to strengthen the project, not something to hide.')


# Engineering appendix — canonical application source

The analysis and visual evidence come first. The cells below preserve additional canonical Python from this project for reviewers who want to inspect pipelines, APIs, tests, feature code, monitoring and reusable implementation details.


## Canonical source: `run.py`


In [ ]:
from __future__ import annotations

import io
import json
import random
import urllib.request
import zipfile
from dataclasses import asdict, dataclass
from pathlib import Path
from typing import Any

import joblib
import numpy as np
import pandas as pd
import torch
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    average_precision_score,
    brier_score_loss,
    f1_score,
    log_loss,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from torch import nn
from torch.utils.data import DataLoader, TensorDataset

ROOT = Path(__file__).resolve().parent
DATA = ROOT / "data"
RESULTS = ROOT / "results"
ARTIFACTS = ROOT / "artifacts"
for folder in (DATA, RESULTS, ARTIFACTS):
    folder.mkdir(exist_ok=True)

DATA_URL = "https://archive.ics.uci.edu/ml/machine-learning-databases/00222/bank-additional.zip"
SEED = 42
TARGET = "y"
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")


def set_seed(seed: int = SEED) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


@dataclass
class ScoreCard:
    roc_auc: float
    pr_auc: float
    f1: float
    precision: float
    recall: float
    brier: float
    log_loss: float
    threshold: float
    positive_rate: float


def download_data(force: bool = False) -> Path:
    target = DATA / "bank-additional-full.csv"
    if target.exists() and not force:
        return target
    with urllib.request.urlopen(DATA_URL, timeout=60) as response:
        payload = response.read()
    with zipfile.ZipFile(io.BytesIO(payload)) as archive:
        member = next(name for name in archive.namelist() if name.endswith("bank-additional-full.csv"))
        target.write_bytes(archive.read(member))
    return target


def load_data(path: Path | None = None) -> pd.DataFrame:
    path = path or download_data()
    frame = pd.read_csv(path, sep=";")
    frame.columns = [str(c).strip().lower().replace(".", "_") for c in frame.columns]
    frame[TARGET] = frame[TARGET].map({"yes": 1, "no": 0}).astype(int)
    return frame


def audit_data(frame: pd.DataFrame) -> dict[str, Any]:
    required = {
        "age", "job", "marital", "education", "default", "housing", "loan", "contact",
        "month", "day_of_week", "duration", "campaign", "pdays", "previous", "poutcome",
        "emp_var_rate", "cons_price_idx", "cons_conf_idx", "euribor3m", "nr_employed", TARGET,
    }
    missing = sorted(required - set(frame.columns))
    if missing:
        raise ValueError(f"Missing columns: {missing}")
    if not set(frame[TARGET].unique()).issubset({0, 1}):
        raise ValueError("Target must be binary")
    if frame[TARGET].nunique() != 2:
        raise ValueError("Both target classes are required")
    return {
        "rows": int(len(frame)),
        "columns": int(frame.shape[1]),
        "duplicate_rows": int(frame.duplicated().sum()),
        "missing_cells": int(frame.isna().sum().sum()),
        "positive_rate": float(frame[TARGET].mean()),
        "age_min": int(frame["age"].min()),
        "age_max": int(frame["age"].max()),
    }


def prepare_frame(frame: pd.DataFrame) -> pd.DataFrame:
    out = frame.drop_duplicates().copy()
    # `duration` is only known after the call completes and is therefore removed
    # for a pre-contact targeting application.
    if "duration" in out.columns:
        out = out.drop(columns=["duration"])
    out["was_previously_contacted"] = (out["pdays"] != 999).astype(int)
    out["previous_success"] = (out["poutcome"] == "success").astype(int)
    out["campaign_intensity"] = np.log1p(out["campaign"].clip(lower=0))
    out["macro_pressure"] = out["euribor3m"] * out["emp_var_rate"]
    return out


def temporal_like_split(frame: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    # UCI release does not expose a full event timestamp. We preserve order and
    # use contiguous blocks to avoid random mixing across the source sequence.
    n = len(frame)
    train_end = int(n * 0.70)
    val_end = int(n * 0.85)
    train = frame.iloc[:train_end].copy()
    validation = frame.iloc[train_end:val_end].copy()
    test = frame.iloc[val_end:].copy()
    if min(len(train), len(validation), len(test)) == 0:
        raise ValueError("Split produced an empty partition")
    return train, validation, test


def feature_groups(frame: pd.DataFrame) -> tuple[list[str], list[str]]:
    features = [c for c in frame.columns if c != TARGET]
    numeric = [c for c in features if pd.api.types.is_numeric_dtype(frame[c])]
    categorical = [c for c in features if c not in numeric]
    return numeric, categorical


def build_preprocessor(numeric: list[str], categorical: list[str]) -> ColumnTransformer:
    return ColumnTransformer(
        transformers=[
            ("numeric", StandardScaler(), numeric),
            ("categorical", OneHotEncoder(handle_unknown="ignore", sparse_output=False), categorical),
        ],
        remainder="drop",
        verbose_feature_names_out=False,
    )


def transform_splits(train: pd.DataFrame, validation: pd.DataFrame, test: pd.DataFrame):
    numeric, categorical = feature_groups(train)
    preprocessor = build_preprocessor(numeric, categorical)
    x_train = preprocessor.fit_transform(train.drop(columns=[TARGET])).astype(np.float32)
    x_validation = preprocessor.transform(validation.drop(columns=[TARGET])).astype(np.float32)
    x_test = preprocessor.transform(test.drop(columns=[TARGET])).astype(np.float32)
    y_train = train[TARGET].to_numpy(dtype=np.float32)
    y_validation = validation[TARGET].to_numpy(dtype=np.float32)
    y_test = test[TARGET].to_numpy(dtype=np.float32)
    return preprocessor, x_train, y_train, x_validation, y_validation, x_test, y_test


class MarketingMLP(nn.Module):
    def __init__(self, input_dim: int):
        super().__init__()
        self.network = nn.Sequential(
            nn.Linear(input_dim, 256),
            nn.BatchNorm1d(256),
            nn.GELU(),
            nn.Dropout(0.30),
            nn.Linear(256, 128),
            nn.BatchNorm1d(128),
            nn.GELU(),
            nn.Dropout(0.25),
            nn.Linear(128, 64),
            nn.GELU(),
            nn.Dropout(0.15),
            nn.Linear(64, 1),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.network(x).squeeze(1)


def make_loader(x: np.ndarray, y: np.ndarray, batch_size: int, shuffle: bool) -> DataLoader:
    dataset = TensorDataset(torch.from_numpy(x), torch.from_numpy(y))
    generator = torch.Generator().manual_seed(SEED)
    return DataLoader(dataset, batch_size=batch_size, shuffle=shuffle, generator=generator)


def predict_probabilities(model: nn.Module, x: np.ndarray, batch_size: int = 2048) -> np.ndarray:
    model.eval()
    loader = DataLoader(TensorDataset(torch.from_numpy(x)), batch_size=batch_size, shuffle=False)
    outputs: list[np.ndarray] = []
    with torch.no_grad():
        for (features,) in loader:
            logits = model(features.to(DEVICE))
            probabilities = torch.sigmoid(logits).cpu().numpy()
            outputs.append(probabilities)
    return np.concatenate(outputs)


def choose_threshold(y_true: np.ndarray, probabilities: np.ndarray, max_contact_rate: float = 0.25) -> float:
    candidates = np.linspace(0.05, 0.95, 181)
    best_threshold = 0.5
    best_f1 = -1.0
    for threshold in candidates:
        pred = (probabilities >= threshold).astype(int)
        contact_rate = float(pred.mean())
        if contact_rate > max_contact_rate or pred.sum() == 0:
            continue
        score = f1_score(y_true, pred, zero_division=0)
        if score > best_f1:
            best_f1 = float(score)
            best_threshold = float(threshold)
    return best_threshold


def score(y_true: np.ndarray, probabilities: np.ndarray, threshold: float) -> ScoreCard:
    clipped = np.clip(probabilities, 1e-6, 1 - 1e-6)
    prediction = (clipped >= threshold).astype(int)
    return ScoreCard(
        roc_auc=float(roc_auc_score(y_true, clipped)),
        pr_auc=float(average_precision_score(y_true, clipped)),
        f1=float(f1_score(y_true, prediction, zero_division=0)),
        precision=float(precision_score(y_true, prediction, zero_division=0)),
        recall=float(recall_score(y_true, prediction, zero_division=0)),
        brier=float(brier_score_loss(y_true, clipped)),
        log_loss=float(log_loss(y_true, clipped)),
        threshold=float(threshold),
        positive_rate=float(prediction.mean()),
    )


def train_neural_network(
    x_train: np.ndarray,
    y_train: np.ndarray,
    x_validation: np.ndarray,
    y_validation: np.ndarray,
    max_epochs: int = 60,
) -> tuple[MarketingMLP, pd.DataFrame]:
    model = MarketingMLP(x_train.shape[1]).to(DEVICE)
    positives = max(float(y_train.sum()), 1.0)
    negatives = max(float(len(y_train) - y_train.sum()), 1.0)
    pos_weight = torch.tensor([negatives / positives], dtype=torch.float32, device=DEVICE)
    criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
    optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="max", factor=0.5, patience=3)
    train_loader = make_loader(x_train, y_train, batch_size=512, shuffle=True)

    best_state: dict[str, torch.Tensor] | None = None
    best_auc = -np.inf
    patience = 8
    stale_epochs = 0
    history: list[dict[str, float | int]] = []

    for epoch in range(1, max_epochs + 1):
        model.train()
        losses: list[float] = []
        for features, target in train_loader:
            features = features.to(DEVICE)
            target = target.to(DEVICE)
            optimizer.zero_grad(set_to_none=True)
            logits = model(features)
            loss = criterion(logits, target)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)
            optimizer.step()
            losses.append(float(loss.detach().cpu()))

        validation_probability = predict_probabilities(model, x_validation)
        validation_auc = float(roc_auc_score(y_validation, validation_probability))
        scheduler.step(validation_auc)
        current_lr = float(optimizer.param_groups[0]["lr"])
        history.append({
            "epoch": epoch,
            "train_loss": float(np.mean(losses)),
            "validation_roc_auc": validation_auc,
            "learning_rate": current_lr,
        })

        if validation_auc > best_auc + 1e-4:
            best_auc = validation_auc
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            stale_epochs = 0
        else:
            stale_epochs += 1
        if stale_epochs >= patience:
            break

    if best_state is None:
        raise RuntimeError("Training failed to produce a checkpoint")
    model.load_state_dict(best_state)
    model.to(DEVICE)
    return model, pd.DataFrame(history)


def fit_logistic_baseline(x_train: np.ndarray, y_train: np.ndarray) -> LogisticRegression:
    model = LogisticRegression(max_iter=2000, class_weight="balanced", n_jobs=-1)
    model.fit(x_train, y_train.astype(int))
    return model


def probability_bins(y_true: np.ndarray, probabilities: np.ndarray, bins: int = 10) -> pd.DataFrame:
    frame = pd.DataFrame({"target": y_true, "probability": probabilities})
    frame["bin"] = pd.qcut(frame["probability"], q=bins, duplicates="drop")
    return (
        frame.groupby("bin", observed=True)
        .agg(rows=("target", "size"), predicted_rate=("probability", "mean"), observed_rate=("target", "mean"))
        .reset_index()
        .assign(bin=lambda x: x["bin"].astype(str))
    )


def segment_errors(frame: pd.DataFrame, probabilities: np.ndarray, threshold: float) -> pd.DataFrame:
    work = frame[["age", "job", "contact", TARGET]].copy().reset_index(drop=True)
    work["probability"] = probabilities
    work["prediction"] = (work["probability"] >= threshold).astype(int)
    work["correct"] = (work["prediction"] == work[TARGET]).astype(int)
    work["age_band"] = pd.cut(work["age"], [0, 29, 39, 49, 59, 200], labels=["<30", "30s", "40s", "50s", "60+"])
    outputs = []
    for column in ["age_band", "contact", "job"]:
        grouped = work.groupby(column, observed=True).agg(
            rows=(TARGET, "size"),
            response_rate=(TARGET, "mean"),
            predicted_probability=("probability", "mean"),
            accuracy=("correct", "mean"),
        ).reset_index()
        grouped.insert(0, "slice", column)
        grouped.rename(columns={column: "value"}, inplace=True)
        outputs.append(grouped)
    return pd.concat(outputs, ignore_index=True)


def targeting_policy(probability: float, threshold: float) -> dict[str, Any]:
    if probability >= max(threshold, 0.70):
        tier = "priority"
        action = "include in high-priority outreach queue"
    elif probability >= threshold:
        tier = "eligible"
        action = "include if campaign capacity remains"
    else:
        tier = "hold"
        action = "do not target in this campaign"
    return {"probability": float(probability), "threshold": float(threshold), "tier": tier, "action": action}


def save_checkpoint(model: MarketingMLP, input_dim: int) -> None:
    torch.save({"state_dict": model.state_dict(), "input_dim": int(input_dim)}, ARTIFACTS / "marketing_mlp.pt")


def main() -> None:
    set_seed()
    raw = load_data()
    audit = audit_data(raw)
    data = prepare_frame(raw)
    train, validation, test = temporal_like_split(data)

    preprocessor, x_train, y_train, x_validation, y_validation, x_test, y_test = transform_splits(train, validation, test)
    baseline = fit_logistic_baseline(x_train, y_train)
    baseline_validation_probability = baseline.predict_proba(x_validation)[:, 1]
    baseline_threshold = choose_threshold(y_validation, baseline_validation_probability)
    baseline_score = score(y_validation, baseline_validation_probability, baseline_threshold)

    model, history = train_neural_network(x_train, y_train, x_validation, y_validation)
    validation_probability = predict_probabilities(model, x_validation)
    threshold = choose_threshold(y_validation, validation_probability)
    validation_score = score(y_validation, validation_probability, threshold)
    test_probability = predict_probabilities(model, x_test)
    test_score = score(y_test, test_probability, threshold)

    calibration = probability_bins(y_test, test_probability)
    slices = segment_errors(test, test_probability, threshold)
    history.to_csv(RESULTS / "training_history.csv", index=False)
    calibration.to_csv(RESULTS / "calibration_bins.csv", index=False)
    slices.to_csv(RESULTS / "error_slices.csv", index=False)
    pd.DataFrame({
        "target": y_test.astype(int),
        "probability": test_probability,
        "prediction": (test_probability >= threshold).astype(int),
    }).to_csv(RESULTS / "test_predictions.csv", index=False)

    joblib.dump(preprocessor, ARTIFACTS / "preprocessor.joblib")
    joblib.dump(baseline, ARTIFACTS / "logistic_baseline.joblib")
    save_checkpoint(model, x_train.shape[1])

    example_probability = float(test_probability[0])
    payload = {
        "dataset_audit": audit,
        "split_rows": {"train": len(train), "validation": len(validation), "test": len(test)},
        "preprocessed_input_dimension": int(x_train.shape[1]),
        "device": str(DEVICE),
        "logistic_validation": asdict(baseline_score),
        "neural_network_validation": asdict(validation_score),
        "neural_network_test": asdict(test_score),
        "roc_auc_gain_vs_logistic": float(validation_score.roc_auc - baseline_score.roc_auc),
        "epochs_trained": int(len(history)),
        "example_targeting_decision": targeting_policy(example_probability, threshold),
        "limitations": [
            "Historical bank campaign data; not representative of every market or current behaviour.",
            "This model is for marketing-response prioritisation, not lending or eligibility decisions.",
            "A real deployment requires consent/privacy review, calibrated campaign costs and ongoing drift checks.",
        ],
    }
    (RESULTS / "metrics.json").write_text(json.dumps(payload, indent=2), encoding="utf-8")
    print(json.dumps(payload, indent=2))


if __name__ == "__main__":
    main()


## Canonical source: `tests/test_neural_network.py`


In [ ]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd
import torch

PROJECT = Path(__file__).resolve().parents[1]
sys.path.insert(0, str(PROJECT))

from run import MarketingMLP, choose_threshold, prepare_frame, score


def test_network_output_shape():
    model = MarketingMLP(input_dim=20)
    batch = torch.randn(8, 20)
    assert model(batch).shape == (8,)


def test_duration_is_removed_for_pre_contact_use():
    frame = pd.DataFrame({
        "duration": [100, 200],
        "pdays": [999, 3],
        "poutcome": ["nonexistent", "success"],
        "campaign": [1, 2],
        "euribor3m": [1.2, 1.3],
        "emp_var_rate": [0.1, 0.2],
        "y": [0, 1],
    })
    prepared = prepare_frame(frame)
    assert "duration" not in prepared.columns
    assert "was_previously_contacted" in prepared.columns


def test_threshold_respects_contact_constraint():
    truth = np.array([0, 0, 0, 1, 1, 1, 0, 0], dtype=float)
    probabilities = np.array([0.01, 0.03, 0.05, 0.90, 0.80, 0.60, 0.20, 0.10])
    threshold = choose_threshold(truth, probabilities, max_contact_rate=0.50)
    result = score(truth, probabilities, threshold)
    assert 0.0 <= threshold <= 1.0
    assert result.positive_rate <= 0.50 + 1e-9


# Portfolio depth check

**Meaningful code lines visible in this notebook:** 549. For a major recruiter-facing application the working target is roughly **1,000 meaningful lines**, with a practical guide of about 600–1,400 depending on the problem. This notebook is below the major-project guide and should grow only through substantive analysis/application depth.

Line count is not a quality metric by itself. Add code only when it improves the real project: data acquisition, validation, cleaning, EDA, visualisation, feature engineering, baselines, model comparison, tuning, leakage control, error analysis, explainability, uncertainty, inference, tests, monitoring, deployment or decision logic.
